In [ ]:
# Fix protobuf version mismatch for kaggle_benchmarks
!pip install protobuf==5.29.6 --quiet


# Trinity Attentional Gateway Probe — Divided Attention

**Task 15 of 25** | Track 3: Attention | Brain Zone: COERULEUS

This notebook measures a model's ability to handle multiple simultaneous streams.

## Trinity Neuroanatomical Foundation

The **Coeruleus** in Trinity manages arousal for parallel processing:

```zig
// src/storm/zones/ofc.zig (Arousal management via toxicity)
pub const ArousalLevel = enum {
    relaxed,
    alert,
    stressed  // Arousal for divided attention
};
```

### Ternary Scoring {-1, 0, +1}

- **+1 (correct)**: Model handles both streams correctly
- **0 (partial)**: Model shows uncertainty about handling
- **-1 (wrong)**: Model fails one or both streams

### φ-Scaling

Dual-task cost follows gradient: 1.0, 1.2, 1.5, 1.8, 2.0

In [ ]:
# Install Kaggle Benchmarks SDK
!pip install -q kaggle-benchmarks

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
from dataclasses import dataclass
from typing import Literal

# Load generated dataset
df = pd.read_csv('../../data/tagp_attention.csv')
print(f"Loaded {len(df)} items")
df.head()

In [ ]:
# Filter for divided attention task
divided_df = df[df['task'] == 'Divided Attention'].copy()
print(f"Divided attention items: {len(divided_df)}")
divided_df[['id', 'task', 'context', 'query', 'difficulty']].head()

In [ ]:
# Define structured output for divided attention
@dataclass
class DividedAttentionResponse:
    """Model's response to divided attention task."""
    stream1_answer: str  # Answer for first query
    stream2_answer: str  # Answer for second query
    confidence: float  # 0.0 to 1.0
    reported_cost: float  # Self-reported dual-task cost
    
    def ternary_score(self, expected_1: str, expected_2: str) -> Literal[-1, 0, 1]:
        """Return Trinity ternary score {-1, 0, +1}."""
        # Correct: both answers correct
        if self.stream1_answer == expected_1 and self.stream2_answer == expected_2:
            return 1
        # Partial: one correct with uncertainty
        elif (self.stream1_answer == expected_1 or self.stream2_answer == expected_2) and 0.3 < self.confidence < 0.8:
            return 0
        # Wrong: both wrong or overconfident
        else:
            return -1

print("DividedAttentionResponse schema defined")

In [ ]:
# Define Kaggle benchmark task
@kbench.task(name="trinity_coeruleus_divided_attention")
def divided_attention(
    llm: kbench.LLM,
    context: str,
    query_1: str,
    query_2: str,
    expected_1: str,
    expected_2: str,
    dual_cost: float
) -> float:
    """
    Measure model's divided attention ability.
    
    Returns:
        Divided attention score: 1.0 (perfect) to -1.0 (worst)
    """
    prompt = f"""You need to handle two simultaneous streams.

Context: {context}

Stream 1 Query: {query_1}
Stream 2 Query: {query_2}

Dual-task cost: {dual_cost}

Provide:
1. Answer for Stream 1
2. Answer for Stream 2
3. Your confidence in handling both streams (0.0 to 1.0)
"""
    
    response = llm.prompt(
        prompt,
        schema=DividedAttentionResponse
    )
    
    # Calculate ternary score
    ternary = response.ternary_score(expected_1, expected_2)
    
    # Combine: 40% accuracy, 30% confidence, 30% cost accuracy
    accuracy_1 = 1.0 if response.stream1_answer == expected_1 else -1.0
    accuracy_2 = 1.0 if response.stream2_answer == expected_2 else -1.0
    combined_accuracy = (accuracy_1 + accuracy_2) / 2
    confidence_score = (response.confidence - 0.5) * 2  # [-1, 1]
    cost_accuracy = 1.0 - abs(response.reported_cost - dual_cost) / dual_cost  # [0, 1]
    
    final_score = 0.4 * combined_accuracy + 0.3 * confidence_score + 0.3 * cost_accuracy
    
    return max(-1.0, min(1.0, final_score))

print("Task 'trinity_coeruleus_divided_attention' registered")

In [ ]:
# Run evaluation on a sample
sample_df = divided_df.head(10)  # Test with 10 items first

results = divided_attention.evaluate(
    llm=[kbench.llm],  # Default test LLM
    evaluation_data=sample_df
)

print("Evaluation Results (Sample):")
print(f"Mean Score: {results['score'].mean():.4f}")
print(f"Std Dev: {results['score'].std():.4f}")
print(f"Min: {results['score'].min():.4f}")
print(f"Max: {results['score'].max():.4f}")
results.head()

In [ ]:
# Full evaluation (uncomment when ready)
# results = divided_attention.evaluate(
#     llm=[kbench.llm],
#     evaluation_data=divided_df
# )
# 
# print(f"\nFull Evaluation Results ({len(divided_df)} items):")
# print(f"Mean Score: {results['score'].mean():.4f}")
# print(f"Ternary Distribution:")
# print(results['ternary_outcome'].value_counts())

In [ ]:
# Submit to Kaggle leaderboard
# Uncomment and run when ready to submit
# kbench.submit(
#     task=divided_attention,
#     results=results,
#     message="Trinity Coeruleus Divided Attention v1.0"
# )
# print("✅ Submitted to Kaggle leaderboard!")

## Expected Leaderboard Performance

Models are expected to show clear separation on this benchmark:

| Model | Expected Score | Reason |
|-------|---------------|--------|
| GPT-4o | 0.60-0.70 | Good divided attention |
| Claude Sonnet | 0.55-0.65 | Moderate divided processing |
| Gemini Pro | 0.50-0.60 | Weak divided attention |
| Llama 3 70B | 0.35-0.45 | Poor divided attention |

### Dual-Task Cost vs Performance

Expected performance degradation:
- Cost 1.0: 85% accuracy
- Cost 1.5: 75% accuracy
- Cost 1.8: 65% accuracy
- Cost 2.0: 55% accuracy